In [3]:
from pathlib import Path
import ast
import pandas as pd

input_path = Path("data/raw/dev.parquet")
output_path = Path("data/raw/dev_context.txt")

df = pd.read_parquet(input_path)


def format_context_item(title: str, sentences: list[str]) -> str:
    lines = [title]
    for sentence in sentences:
        lines.append(f"  {sentence}")
    return "\n".join(lines)


def normalize_text(value: str) -> str:
    return " ".join(str(value).split())


blocks = []
seen_context_items = set()
for index, row in df.iterrows():
    context_items = ast.literal_eval(row["context"])
    for title, sentences in context_items:
        context_key = (
            normalize_text(title),
            tuple(normalize_text(sentence) for sentence in sentences),
        )
        if context_key in seen_context_items:
            continue
        seen_context_items.add(context_key)
        blocks.append(format_context_item(title, sentences))

output_path.write_text("\n\n".join(blocks), encoding="utf-8", errors="replace")
print(f"Saved {len(blocks)} unique items to {output_path}")

Saved 56684 unique items to data/raw/dev_context.txt
